# Testing linear attention blocks

Get root.

In [ ]:
import os, sys, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import importlib
from torch.nn.attention.flex_attention import flex_attention
from pathlib import Path
!pip install linear-attention-transformer

# Working repo
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
root = Path("/content/mini-gLM")
if repo_dir.exists():
    subprocess.run(["rm", "-r", root])
subprocess.run(["git", "clone", repo_url])
sys.path.insert(0, str(root))

# Torch configs
torch.manual_seed(111)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
flex_attention = torch.compile(flex_attention, dynamic = True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Synthetic data.

In [ ]:
# Configs
B = 8
L = 200
d_model = 256
num_heads = 8
vocab_size = 512
p_drop = 0.1
vocab_size = 512
max_seq_len = 1000
num_blocks = 6

# Synthetic data
x = torch.randint(low = 0, high = 512, size = (B, L)).long().to(device)
embed = torch.randn(B, L, D).float().to(device)
probs = torch.full((B, L), 0.5).to(device)
attn_mask = torch.bernoulli(probs).bool().to(device)

Test linear attention transformer.

In [ ]:
from src.model import LinearGLM

model = LinearGLM(
    num_blocks = num_blocks,
    d_model = d_model,
    num_heads = num_heads,
    vocab_size = vocab_size,
    p_drop = p_drop,
    max_seq_len = max_seq_len
).to(device)

model(x, attn_mask = attn_mask)

Try to integrate alternating dense and linear attention blocks.